# 10 — ML: Segmentación de Zonas (K-Means) — Dashboard 8

**Objetivo:** Segmentar automáticamente las zonas de taxi en perfiles operativos.

**Justificación académica:**
K-Means agrupa zonas por similitud de comportamiento (demanda, ingresos, hora pico y propinas).
El resultado permite responder: *'¿Qué tipo de zona es el Upper East Side comparado con el Aeropuerto JFK?'*

**Flujo:**
1. Lee `mart_financial_performance` y `mart_demand_volume` desde `tlc_gold`.
2. Construye un Feature Vector por zona (8 características).
3. Aplica K-Means con selección óptima de K via Elbow Method.
4. Interpreta y etiqueta los clusters.
5. Persiste resultados en `tlc_gold.ml_zone_segments`.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
import pyspark.sql.functions as F

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import joblib
from pathlib import Path

MODELS_DIR = Path('/home/jovyan/work/data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = Path('/home/jovyan/work/data/charts')
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK.')

Imports OK.


In [2]:
spark = get_spark('TLC_ML_Segmentation')
print(f'Spark {spark.version} ready.')

2026-07-19 16:35:34 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
Spark 3.4.1 ready.


In [3]:
# ── Leer Data Marts ya agregados (ligeros, pocos miles de filas)
df_fin  = read_mongo(spark, settings.MONGO_DB_GOLD, 'mart_financial_performance')
df_vol  = read_mongo(spark, settings.MONGO_DB_GOLD, 'mart_demand_volume')
df_abc  = read_mongo(spark, settings.MONGO_DB_GOLD, 'mart_abc_xyz_zones')

# ── Feature Engineering por Zona
# Agregar características financieras por zona
zone_fin = (
    df_fin
    .groupBy('pickup_zone_id', 'zone_name', 'borough')
    .agg(
        F.sum('total_trips').alias('total_trips'),
        F.sum('total_revenue').alias('total_revenue'),
        F.avg('avg_tip').alias('avg_tip'),
        F.avg('tip_rate_pct').alias('tip_rate_pct'),
        F.avg('avg_revenue_per_trip').alias('avg_fare'),
    )
)

# Agregar características de demanda por zona
zone_vol = (
    df_vol
    .groupBy('pickup_zone_id')
    .agg(
        F.avg('avg_duration_min').alias('avg_duration_min'),
        F.avg('avg_distance_miles').alias('avg_distance_miles'),
        # Hora pico (moda de hora con mayor demanda por zona)
        F.first(
            F.col('hour'), ignorenulls=True
        ).alias('peak_hour_approx'),
    )
)

# Join de features
zone_features_spark = (
    zone_fin
    .join(zone_vol, on='pickup_zone_id', how='inner')
)

# Pasar a Pandas (pocas filas = 265 zonas)
zone_features_pd = zone_features_spark.toPandas()
print(f'Zonas con features: {len(zone_features_pd)}')
print(zone_features_pd.describe().round(2))

2026-07-19 16:35:35 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_gold.mart_financial_performance
2026-07-19 16:35:36 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_gold.mart_demand_volume
2026-07-19 16:35:36 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_gold.mart_abc_xyz_zones
Zonas con features: 264
       pickup_zone_id  total_trips  total_revenue  avg_tip  tip_rate_pct  \
count          264.00       264.00   2.640000e+02   264.00        264.00   
mean           133.11   3143669.38   9.817840e+07     1.04          0.16   
std             76.77   3064103.08   1.517196e+08     0.72          0.07   
min              1.00         2.00   0.000000e+00     0.00          0.00   
25%             66.75   1010334.00   2.285415e+07     0.56          0.10   
50%            133.50   2192048.50   5.282170e+07     0.92          0.15   
75%            199.25   4493680.50   1.233199e+08     1.21          0.22   
max            265.00  18526525.00   1.559487

In [4]:
# ── Preparar features para K-Means
FEATURE_COLS = [
    'total_trips', 'total_revenue', 'avg_tip', 'tip_rate_pct',
    'avg_fare', 'avg_duration_min', 'avg_distance_miles', 'peak_hour_approx',
]

X_raw = zone_features_pd[FEATURE_COLS].fillna(0)

# Escalar (K-Means es sensible a magnitudes)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Feature matrix shape: {X_scaled.shape}')

Feature matrix shape: (264, 8)


In [5]:
# ── Elbow Method: encontrar K óptimo
inertias       = []
silhouette_scores = []
K_RANGE        = range(2, 9)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    score = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(score)
    print(f'  K={k} | Inertia={km.inertia_:,.0f} | Silhouette={score:.3f}')

# Gráfica Elbow
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertias, 'bo-', linewidth=2)
axes[0].set_title('Elbow Method — Inercia por K', fontweight='bold')
axes[0].set_xlabel('K (Número de Clusters)')
axes[0].set_ylabel('Inercia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_RANGE), silhouette_scores, 'go-', linewidth=2)
axes[1].set_title('Silhouette Score por K', fontweight='bold')
axes[1].set_xlabel('K (Número de Clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'kmeans_01_elbow.png'), dpi=120)
plt.close()
print('Gráfica Elbow guardada.')

  K=2 | Inertia=1,779 | Silhouette=0.314
  K=3 | Inertia=1,438 | Silhouette=0.225
  K=4 | Inertia=1,257 | Silhouette=0.215
  K=5 | Inertia=1,028 | Silhouette=0.225
  K=6 | Inertia=901 | Silhouette=0.216
  K=7 | Inertia=802 | Silhouette=0.224
  K=8 | Inertia=747 | Silhouette=0.218
Gráfica Elbow guardada.


In [6]:
# ── Entrenar modelo final con K=4 (4 perfiles de zona es interpretable para el negocio)
K_OPTIMAL = 4

kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init='auto')
kmeans_final.fit(X_scaled)

# Asignar clusters al dataframe
zone_features_pd['cluster'] = kmeans_final.labels_

# ── Etiquetar clusters con nombre de negocio (basado en los centroides)
cluster_profiles = (
    zone_features_pd
    .groupby('cluster')[FEATURE_COLS]
    .mean()
    .round(2)
)

print('Perfil promedio por cluster:')
print(cluster_profiles.to_string())

# Etiquetar manualmente basado en los centroides
# (El orden puede variar; revisar cluster_profiles para ajustar si es necesario)
sorted_by_revenue = cluster_profiles['total_revenue'].sort_values(ascending=False)
label_map = {}
labels    = ['Zona Premium (Alto Tráfico)', 'Zona Comercial (Tráfico Medio)', 'Zona Residencial', 'Zona Marginal (Bajo Tráfico)']
for rank, cluster_id in enumerate(sorted_by_revenue.index):
    label_map[cluster_id] = labels[rank]

zone_features_pd['cluster_label'] = zone_features_pd['cluster'].map(label_map)

print('\nDistribución de zonas por cluster:')
print(zone_features_pd.groupby('cluster_label')['pickup_zone_id'].count())

# Persistir modelos
joblib.dump(kmeans_final, MODELS_DIR / 'kmeans_zones.pkl', compress=3)
joblib.dump(scaler, MODELS_DIR / 'kmeans_scaler.pkl', compress=3)
print('\n✓ Modelos guardados.')

Perfil promedio por cluster:
         total_trips  total_revenue  avg_tip  tip_rate_pct  avg_fare  avg_duration_min  avg_distance_miles  peak_hour_approx
cluster                                                                                                                     
0         1949966.35   5.222552e+07     1.03          0.16     27.12             23.72                4.19             11.54
1          584179.67   1.627984e+07     2.65          0.22     42.89             35.60                5.38             13.12
2         2384142.23   5.372574e+07     0.41          0.07     25.50             70.25                4.76             13.53
3         7802074.11   2.868481e+08     1.05          0.22     22.77             27.05                3.54             16.47

Distribución de zonas por cluster:
cluster_label
Zona Comercial (Tráfico Medio)     60
Zona Marginal (Bajo Tráfico)       24
Zona Premium (Alto Tráfico)        55
Zona Residencial                  125
Name: pickup_zone_i

In [7]:
# ── Visualización PCA (reducir a 2D para graficar)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.cumsum()[-1] * 100

colors = ['#ef4444', '#3b82f6', '#10b981', '#f59e0b']
fig, ax = plt.subplots(figsize=(12, 8))

for cluster_id, label in label_map.items():
    mask = zone_features_pd['cluster'] == cluster_id
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors[cluster_id % len(colors)],
        label=f'Cluster {cluster_id}: {label}',
        alpha=0.7, s=80, edgecolors='white', linewidth=0.5
    )

ax.set_title(f'K-Means Segmentación de Zonas NYC (K={K_OPTIMAL})\nPCA — Varianza explicada: {explained:.1f}%',
             fontsize=13, fontweight='bold')
ax.set_xlabel('PCA Componente 1')
ax.set_ylabel('PCA Componente 2')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'kmeans_02_clusters_pca.png'), dpi=150)
plt.close()
print('Gráfica PCA guardada.')

Gráfica PCA guardada.


In [8]:
# ── Guardar resultados en MongoDB Gold
result_df = zone_features_pd[['pickup_zone_id', 'zone_name', 'borough',
                               'cluster', 'cluster_label'] + FEATURE_COLS].copy()

result_spark = spark.createDataFrame(result_df)
write_mongo(result_spark, settings.MONGO_DB_GOLD, 'ml_zone_segments', mode='overwrite')
print(f'✓ {len(result_df)} zonas segmentadas guardadas en tlc_gold.ml_zone_segments')

# Resumen final
print('\n── Distribución final de clusters ──')
print(result_df.groupby('cluster_label')[['total_trips', 'total_revenue']]
      .sum().sort_values('total_revenue', ascending=False)
      .round(0).to_string())

2026-07-19 16:36:19 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.ml_zone_segments (mode=overwrite)
✓ 264 zonas segmentadas guardadas en tlc_gold.ml_zone_segments

── Distribución final de clusters ──
                                total_trips  total_revenue
cluster_label                                             
Zona Premium (Alto Tráfico)       429114076   1.577665e+10
Zona Residencial                  243745794   6.528190e+09
Zona Comercial (Tráfico Medio)    143048534   3.223545e+09
Zona Marginal (Bajo Tráfico)       14020312   3.907162e+08
